<a href="https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/Basic_python_colab_and_fasta_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python in Colab: from a first line of code to a FASTA tool

**For students new to programming · about 90–120 minutes, or two short sessions**

A notebook is a lab record that combines explanations, executable instructions, and results.
Today you will read and change a few lines yourself, then use **Gemini, Claude, or ChatGPT** as a coding partner.
You remain responsible for deciding what the program should do and checking whether it did it.

By the end you should be able to:
- Run a code cell and explain a variable, a function call, and a loop.
- Find a data file using a path and read file paths stored inside a CSV table.
- Ask an AI assistant for a small, clearly specified program.
- Check that a FASTA-header cleaner preserves the sequences and records where each new ID came from.

All sequences here are invented teaching examples, not biological measurements.


[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/#fileId=)

**Version 2 · student edition.** The badge opens Colab’s notebook dialog: choose **Upload** and select this downloaded `.ipynb`. A direct badge for this exact notebook requires its published GitHub URL or shared Drive ID.

**Experiment log:** For each change, record *prediction → one change → observed result → explanation*. Keep a baseline copy of the cell so you can restore it. Blank AI workspaces are intentional: a first Run all executes the provided demonstration; a completed submission also contains your tested AI code.


## 1. Open your own laboratory notebook

1. Open [Google Colab](https://colab.research.google.com/) and upload this `.ipynb` using **File → Upload notebook** (or the upload tab in the open dialog).
2. Save a personal copy in Drive and give it your name. A normal CPU runtime is sufficient.
3. Click the triangular run button next to a code cell, or press **Shift+Enter**. Start at the top and work downward.
4. Double-click a text cell to edit your predictions and answers.

Text cells explain; code cells execute. The result printed below a code cell is its **output**.
The code runs on a remote computer, not on your laptop. Variables and runtime files can disappear when that runtime is deleted.
Saving a notebook does not automatically save every file your code creates; export results at the end.

**Before you run:** What do you expect the next cell to display?  Then change the greeting and one number, rerun, and explain which output changed. Try `"2" + "3"` and compare it with `2 + 3`: what do the quotation marks change? Restore the baseline afterward.


In [2]:
print("Hello, bioinformatics!")
print(2 + 3)


Hello, bioinformatics!
5


### Code is a sequence of instructions

Python evaluates expressions and carries out instructions in order. A **variable** gives a value a name.
`=` assigns a value; it does not ask whether two values are equal. Quotation marks mark text (a **string**).
Numbers have no quotation marks. A comment begins with `#` and explains the code to a human.

`len(...)` is a **function call**: it receives something as input and returns its length.
`print(...)` displays a value. A method such as `sequence.count("G")` is a function attached to an object.
Predict the length below, then run the cell.


In [3]:
sequence = "ATGCGGGTA"
sequence_length = len(sequence)
g_count = sequence.count("G")
print("Sequence:", sequence)
print("Length:", sequence_length)
print("Number of G bases:", g_count)


Sequence: ATGCGGGTA
Length: 9
Number of G bases: 4


**Try it yourself.** Change the sequence above to `AAAA`, rerun that cell, and explain what changed.
Then restore `ATGCGGGTA` and rerun. If you change code without running it, Python still remembers the old value.

**Your explanation:** _Replace this sentence._


### Lists, indexing, and a repeated instruction

A **list** keeps several values in order. Python counts positions starting at **0**.
A `for` loop repeats the indented instructions once for each item in a collection.
The colon starts the loop body; indentation shows which lines belong to it.


In [11]:
sequences = ["ATGC", "GGCCAA", "ATATATAT"]
print("First sequence:", sequences[0])
print("Number of sequences:", len(sequences))

for dna in sequences:
    print(dna, "has", len(dna), "bases")
print("The loop is finished.")


First sequence: ATGC
Number of sequences: 3
ATGC has 4 bases
GGCCAA has 6 bases
ATATATAT has 8 bases
The loop is finished.


**Small exercise without AI:** Add a fourth sequence to the list and rerun. Predict how many times the indented line runs.

A **condition** lets code choose what to do. `>=` means “greater than or equal to”; `==` tests equality.
Read this cell aloud: “For each sequence, if its length is at least six…”


In [ ]:
for dna in sequences:
    if len(dna) >= 6:
        print("Keep:", dna)


**Your answers:** How many sequences passed the length test? What would change if you used `> 6`?

_Replace this sentence._


## 2. Give your notebook a data folder

A **library** provides code other people have already written. `import` makes it available.
The next cell is provided setup: run it before continuing; you do not need to memorize it.
It creates three tiny files so everyone can start without downloads or Drive permissions.
Rerunning it preserves existing files. To reset an edited practice file, remove that practice copy in the Files panel and rerun setup.

`Path.cwd()` gives the current working directory. With `Path` objects, `/` joins folder and file names.
`mkdir(...)` creates a folder. Here `parents=True` creates missing parent folders and `exist_ok=True` allows an existing folder.


In [ ]:
from pathlib import Path
import csv

# Relative to the runtime's current working directory.
PROJECT_DIR = Path.cwd() / "python_bio_lab"
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Small synthetic teaching data: no external download or account is needed.
teaching_files = {
    "messy_sequences.fasta": '>sample A | gene=psbA | organism=teaching_example\nATGCGT\nAATT\n>sample A | gene=psbD | replicate=2\nGGCC\nATAT\n\n>  lake-B description with extra spaces\nacgtNN\nACGT\n>river|C hypothetical protein\nATATAT\nAT\n',
    "pond_sequences.fasta": '>pond 1 | gene=example_A\nGCGCGC\nATAT\n>pond 2 | gene=example_B\nAATT\nGGCCNN\n',
    "samples.csv": 'sample_id,fasta_path,habitat\nlesson_A,messy_sequences.fasta,lake\nlesson_B,pond_sequences.fasta,pond\n',
}
for filename, contents in teaching_files.items():
    destination = DATA_DIR / filename
    if not destination.exists():  # Preserve any file you have already edited.
        destination.write_text(contents, encoding="utf-8")

print("Working directory:", Path.cwd())
print("Data folder:", DATA_DIR)
print("Results folder:", RESULTS_DIR)


### A path is an address, not the data itself

`DATA_DIR / "messy_sequences.fasta"` names a location. `.read_text()` opens the file at that location and reads its contents.
A **relative path** such as `data/example.fasta` needs a starting folder. An **absolute path** begins at the filesystem root.

In Colab, a path copied from your Mac or Windows computer will not exist on the remote runtime. Use the Files panel to inspect the runtime's folders.
Predict which output below will be a path and which will contain DNA.


In [ ]:
fasta_path = DATA_DIR / "messy_sequences.fasta"
print("File path:", fasta_path)
print("Does it exist?", fasta_path.exists())
fasta_text = fasta_path.read_text(encoding="utf-8")
print("File contents:\n", fasta_text)


### Your path experiment — define once, reuse twice

Before running, predict which records you will see when you select the pond file. Define `practice_path` once below and use it for both checks. Change only the filename to `messy_sequences.fasta`, rerun, and compare. Why is a named path easier to maintain than repeating a literal filename?

Next try a deliberately misspelled filename. The check should explain the missing file without attempting to read it. Restore a valid filename. `practice_path` is your experiment variable; `fasta_path` remains the required FASTA input.

**My prediction, change, evidence, and explanation:** _Write here._

In [ ]:
practice_path = DATA_DIR / "pond_sequences.fasta"  # Edit this one filename.
print("Selected path:", practice_path)
if practice_path.is_file():
    print(practice_path.read_text(encoding="utf-8"))
    print("File size in bytes:", practice_path.stat().st_size)
else:
    print("File not found: check the filename and DATA_DIR.")

### FASTA: headers and sequence lines

A FASTA record starts with a header line beginning with `>`. Its sequence can wrap across several following lines.
A file may contain many records. In many tools, only the first whitespace-delimited word of a header is treated as the ID.
Our first two headers would therefore both become `sample`: a collision that can make it difficult to associate downstream results with the correct sequence.

`with ... open(...)` closes the file when the block ends. The loop reads it one line at a time.
`.startswith(">")` returns `True` for header lines. `.rstrip()` removes trailing whitespace for display only.


In [ ]:
header_count = 0
with fasta_path.open(encoding="utf-8") as handle:
    for line in handle:
        if line.startswith(">"):
            header_count = header_count + 1
            print(line.rstrip())
print("Number of records:", header_count)


**Predict and explain:** Why would counting lines give the wrong number of sequences? Why is deleting everything after the first space insufficient to make our IDs unique?

_Replace this sentence._


### Read paths from a data table into an experimental notebook

Researchers often use a **manifest**: a table listing sample names and the location of each sample's data.
CSV means comma-separated values. Its first line names the columns. Here the `fasta_path` column contains filenames relative to the **manifest's folder**.

`csv.DictReader` reads each row as a dictionary: column names are keys, and the entries are values.
`row["fasta_path"]` retrieves a filename; `manifest_path.parent / ...` turns it into a usable path.
This remains correct even when your notebook's working directory differs from the manifest's directory.


In [ ]:
manifest_path = DATA_DIR / "samples.csv"
print(manifest_path.read_text(encoding="utf-8"))

with manifest_path.open(newline="", encoding="utf-8") as handle:
    sample_rows = list(csv.DictReader(handle))

print("First row:", sample_rows[0])
print("Filename stored in that row:", sample_rows[0]["fasta_path"])


In [ ]:
for row in sample_rows:
    input_path = manifest_path.parent / row["fasta_path"]
    print(row["sample_id"], "→", input_path)
    print("Found:", input_path.exists())
    with input_path.open(encoding="utf-8") as handle:
        count = 0
        for line in handle:
            if line.startswith(">"):
                count = count + 1
    print("Records:", count)


**Your task without AI:** Identify the line that converts text stored in the CSV into a file path.
Explain why `Path(row["fasta_path"])` alone might fail. What kind of error would you expect if a filename were misspelled?

_Replace this sentence._


**Debug with AI:** Ask for a hint about a misspelled path, supplying the printed path and error. Ask it to explain the starting directory before suggesting code. Verify its diagnosis using `.exists()`; do not accept a guessed filename.

### Optional: use a file from your computer or Google Drive

The main exercise uses the practice files already created. Skip these options on your first pass.

**Upload:** Set the flag below to `True`, run the cell, and choose a small FASTA file. This changes `fasta_path` for the next exercises; use the supplied practice file for the required submission.
Files uploaded to the runtime are temporary. To return to the practice file, rerun the earlier `fasta_path = DATA_DIR / ...` cell.


In [ ]:
UPLOAD_A_FILE = False
if UPLOAD_A_FILE:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one FASTA file for this optional example.")
    filename = next(iter(uploaded))
    fasta_path = Path.cwd() / filename
    print("Using:", fasta_path)


**Drive:** This optional cell mounts your Drive and shows an example persistent location.
Replace `Bioinformatics/my_sequences.fasta` with your actual folder and filename before switching the flag on.
Mounting requires Colab's authorization dialog. It is not needed for the practice data.


In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_path = Path("/content/drive/MyDrive/Bioinformatics/my_sequences.fasta")
    print("Exists:", drive_path.exists())
    # After confirming the path, you may set: fasta_path = drive_path


## 3. Switch roles: direct an AI coding partner

From here, use **Gemini, Claude, or ChatGPT**, whichever you can access. A separate chat window is fine; no API key is required.
If you do not have access, work with a partner or use the instructor's reference solution after drafting your own specification.

Your workflow for each task:
1. **Specify:** describe the input, output, and what must remain unchanged.
2. **Predict:** write a small example of the expected result before asking for code.
3. **Generate:** request one small function and a plain-language explanation.
4. **Inspect and run:** identify the input and output paths; run on the practice data.
5. **Check and revise:** use evidence to decide whether it worked; send any error and a small example back to your assistant.

For each task, retain your prompt, tool/model name (if shown), date, one revision, and the checks you used.
Use the invented data here in your prompts.

A function defined with `def` packages reusable instructions. Defining it does not run it;
calling its name with arguments runs its body. `return` sends a result back to the caller.


### AI task A — build a traceable FASTA-header cleaner (required)

**Biological purpose:** downstream programs need unambiguous IDs, while you still need to trace a result back to its original record.

Our exact policy for this exercise:
- Give records IDs `seq001`, `seq002`, … in their original order, starting over for each input file.
- Replace each entire header with `>` followed by its new ID.
- Preserve every non-header line, including sequence case, sequence wrapping, blank lines, and record order.
- Write a separate CSV with columns `new_id,original_header`. Retain the full original header excluding `>` and the line ending; retain spaces and punctuation.
- Write new output files and leave the original input unchanged. Reject an output path equal to the input or to the other output path.
- Assume a valid, nonempty UTF-8 FASTA using LF line endings. This is a focused cleaner, not a general sequence-format validator.

**Before prompting:** Write the expected first two new header lines. Why should the two old headers that start with `sample A` get different IDs?

_Your prediction:_

**Starter prompt — adapt it in your own words:**
> I am learning Python in Colab. Write a function `clean_fasta(input_path, output_path, mapping_path)` using only Python's standard library. [Paste the policy above and the small FASTA example.] Use pathlib and csv. Give me one code cell defining the function, without running it. Explain the loop and how you preserve all non-header lines. Do not modify my original file. Include path checks so all three paths are distinct. Keep the code readable for a beginner.

Paste the returned function into the next cell. Remove Markdown code fences such as triple backticks.


In [ ]:
# AI workspace A: replace this comment with the generated clean_fasta function.
# The function must be named clean_fasta and accept the three paths in the prompt.


In [ ]:
cleaned_path = RESULTS_DIR / "cleaned_sequences.fasta"
mapping_path = RESULTS_DIR / "header_mapping.csv"

if callable(globals().get("clean_fasta")):
    original_bytes = fasta_path.read_bytes()
    clean_fasta(fasta_path, cleaned_path, mapping_path)
    assert fasta_path.read_bytes() == original_bytes, "The input file was modified!"
    print(cleaned_path.read_text(encoding="utf-8"))
    print(mapping_path.read_text(encoding="utf-8"))
else:
    print("PENDING: add your clean_fasta function to workspace A, then rerun this cell.")


### Check the result instead of trusting its appearance

The provided helper below collects records so we can compare them. Read its purpose; you do not need to memorize its implementation.
`assert` checks a statement and stops with an error when it is false. A failure is useful evidence, not a grade.
A successful run below checks these specific requirements; it does not prove the program handles every possible FASTA file.


In [ ]:
def read_fasta_records(path):
    """Small inspection helper for this exercise, not a general FASTA validator."""
    records = []
    header = None
    parts = []
    with Path(path).open(encoding="utf-8") as handle:
        for line in handle:
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(parts)))
                header = line[1:].rstrip("\r\n")
                parts = []
            elif line.strip():
                if header is None:
                    raise ValueError("Sequence text appeared before the first header.")
                parts.append(line.strip())
    if header is not None:
        records.append((header, "".join(parts)))
    return records


In [ ]:
def check_cleaning(original_path, cleaned_path, mapping_path):
    """Check our assignment's contract; an assertion failure needs investigation."""
    before = read_fasta_records(original_path)
    after = read_fasta_records(cleaned_path)
    expected_ids = [f"seq{i:03d}" for i in range(1, len(before) + 1)]
    assert len(before) > 0, "The input must contain records."
    assert [h for h, s in after] == expected_ids, "IDs must be seq001, seq002, ... in order."
    assert [s for h, s in before] == [s for h, s in after], "Sequence content or order changed."
    # Also compare every non-header line, including wrapping, case, and blank lines.
    old_lines = Path(original_path).read_text(encoding="utf-8").splitlines(keepends=True)
    new_lines = Path(cleaned_path).read_text(encoding="utf-8").splitlines(keepends=True)
    assert len(old_lines) == len(new_lines), "The number of lines changed."
    for old, new in zip(old_lines, new_lines):
        if not old.startswith(">"):
            assert old == new, "A non-header line changed."
    with Path(mapping_path).open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        assert reader.fieldnames == ["new_id", "original_header"], "Check CSV column names."
        rows = list(reader)
    assert rows == [
        {"new_id": new_id, "original_header": header}
        for new_id, (header, sequence) in zip(expected_ids, before)
    ], "The mapping lost or changed an original header."
    print(f"PASS: {len(before)} records; IDs, sequences, lines, and mapping verified.")


In [ ]:
if callable(globals().get("clean_fasta")) and cleaned_path.exists() and mapping_path.exists():
    check_cleaning(fasta_path, cleaned_path, mapping_path)
else:
    print("PENDING: generate the cleaned FASTA and mapping before running these checks.")


### A second example catches fragile code

This example includes repeated original headers, a comma and quotation marks in a header, lowercase sequence, `N`, and a blank line.
A proper CSV writer handles punctuation without corrupting columns.
Predict how many unique output IDs should appear.


In [ ]:
edge_input = RESULTS_DIR / "edge_input.fasta"
edge_output = RESULTS_DIR / "edge_cleaned.fasta"
edge_mapping = RESULTS_DIR / "edge_mapping.csv"
edge_input.write_text('>dup description, "quoted"\nacgN\nTT\n\n>dup description, "quoted"\nGGcc\n', encoding="utf-8")
if callable(globals().get("clean_fasta")):
    edge_before = edge_input.read_bytes()
    clean_fasta(edge_input, edge_output, edge_mapping)
    assert edge_input.read_bytes() == edge_before, "The edge input changed."
    check_cleaning(edge_input, edge_output, edge_mapping)
else:
    print("PENDING: add the cleaning function first.")


**Record your evidence:** Paste your prompt, the assistant/model and date, an error or revision (or a refinement if the first version passed), and the two check results.
Explain one line of the generated function in your own words.

_Replace this sentence._


### AI task B — design a second small tool (required)

Ask your assistant to write `summarize_fasta(input_path, output_csv)`.
For each record it must write the columns `sequence_id,length,gc_percent,n_count`, in input order.
Use the first whitespace-delimited header token as `sequence_id` (use the cleaned file for the main result).
Length counts all bases including `N`; GC% is **100 × (G + C) / full sequence length**, case-insensitively; `N` stays in the denominator.
Reject empty sequences and any characters other than A, C, G, T, N for this exercise. Write a separate CSV and leave the input unchanged.
Define the function in the next cell; do not include calls inside it.

**Write your own prompt.** Explain why the denominator matters and request an explicit check for a zero-length sequence.
Before running anything, calculate the expected length and GC% of `GCNN`: length = ___, GC% = ___.


In [ ]:
# AI workspace B: paste your summarize_fasta function here.


In [ ]:
summary_path = RESULTS_DIR / "sequence_summary.csv"
if callable(globals().get("summarize_fasta")) and cleaned_path.exists():
    before_summary = cleaned_path.read_bytes()
    summarize_fasta(cleaned_path, summary_path)
    assert cleaned_path.read_bytes() == before_summary, "Summary code changed its input."
    with summary_path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        assert reader.fieldnames == ["sequence_id", "length", "gc_percent", "n_count"]
        summary_rows = list(reader)
    expected_records = read_fasta_records(cleaned_path)
    assert len(summary_rows) == len(expected_records)
    for row, (header, dna) in zip(summary_rows, expected_records):
        upper = dna.upper()
        assert row["sequence_id"] == header.split()[0]
        assert int(row["length"]) == len(dna)
        assert int(row["n_count"]) == upper.count("N")
        expected_gc = 100 * (upper.count("G") + upper.count("C")) / len(upper)
        assert abs(float(row["gc_percent"]) - expected_gc) < 0.01
    # An independent hand-computable test for the agreed denominator.
    gc_test = RESULTS_DIR / "gc_test.fasta"
    gc_csv = RESULTS_DIR / "gc_test.csv"
    gc_test.write_text(">test\nGCNN\n", encoding="utf-8")
    summarize_fasta(gc_test, gc_csv)
    with gc_csv.open(newline="", encoding="utf-8") as handle:
        test_row = list(csv.DictReader(handle))[0]
    assert int(test_row["length"]) == 4
    assert float(test_row["gc_percent"]) == 50.0
    assert int(test_row["n_count"]) == 2
    print("PASS: record summaries and the GCNN example agree with the specification.")
    print(summary_path.read_text(encoding="utf-8"))
else:
    print("PENDING: finish task A and add summarize_fasta to workspace B.")


**Interpret:** Which record has the largest GC%? Does that alone identify its species? Explain.
Record your AI prompt and revision here.

_Replace this sentence._


### AI task C — use the manifest to process multiple files (extension)

Ask your assistant for a loop that reads `samples.csv`, resolves every `fasta_path` relative to the manifest's folder,
and calls your tested cleaner once per row. Write outputs into `RESULTS_DIR / sample_id`, so different samples do not overwrite each other.
Have it report each sample's input path and record count. Keep the source FASTA files unchanged.
For our table, you should process **two files**, containing **four and two records**.
Run `check_cleaning` for each sample.

Our IDs restart at `seq001` in each file. Explain why that works for separate files but would need a new policy before merging their records.

_Your prompt, code explanation, and result:_


In [ ]:
# AI workspace C (optional): paste the manifest-processing loop here.


## 4. Save a reproducible laboratory record

1. Complete the prediction and interpretation cells, and retain the prompts you actually used.
2. Restart the runtime/session and run all cells in order. A complete submission should have no `PENDING` messages for tasks A or B.
3. Save/download your completed `.ipynb`. Run the cell below and download its results ZIP using the Files panel, or the optional download switch.

**Submit:** your completed notebook, cleaned FASTA, header mapping CSV, and sequence-summary CSV.
Your notebook should include the main and edge-example check results, your `GCNN` calculation, and a brief explanation of how a path from the manifest reaches the loop.

The ZIP contains practice inputs and results; the notebook must be saved separately through Colab.


In [ ]:
import shutil
archive_path = Path(shutil.make_archive(str(PROJECT_DIR.parent / "python_bio_lab_results"), "zip", PROJECT_DIR))
print("Results archive:", archive_path)
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive_path))


### If you get stuck

| Symptom | First thing to check |
|---|---|
| `NameError` | Did you run the cell that defines that name? Is the spelling identical? |
| `FileNotFoundError` | Print the path, check `.exists()`, and inspect the Files panel. |
| `IndentationError` | Check the spaces under `for`, `if`, `with`, or `def`. |
| `SyntaxError` after pasting AI code | Remove Markdown code fences; check quotes and colons. |
| An assertion fails | Read the message and compare the smallest input and output that demonstrate it. |
| The old result still appears | Rerun the changed function definition, then its call, then the checks. |

**Exit ticket:** Explain “path,” “loop,” and “function” to a classmate without reading their definitions.
Name one thing an AI assistant cannot establish merely by producing plausible code.

References: [Colab FAQ](https://research.google.com/colaboratory/faq.html),
[Python tutorial](https://docs.python.org/3/tutorial/),
[pathlib](https://docs.python.org/3/library/pathlib.html),
[csv](https://docs.python.org/3/library/csv.html).
